In [1]:
"""
Rigorous verification: is the standard GR perihelion-precession formula
the first-order expansion of the WILL RG exact algebraic precession?

WILL RG (exact):
    Delta phi = 2 pi (1 - tau^2) / (1 - e^2)
    tau^2 = (1 - beta^2)(1 - kappa^2)
    tau_Y^2 = 1 - tau^2 = beta^2 + kappa^2 - beta^2 kappa^2

Reference scale: semi-major axis a, closure beta^2 = kappa^2/2 at a,
                 kappa^2(a) = R_s / a.

GR (first-order PN, recovered by dropping kappa^2 beta^2):
    Delta phi = 3 pi R_s / (a (1 - e^2))
"""

import sympy as sp

# -----------------------------------------------------------
# Symbolic identity
# -----------------------------------------------------------
kappa2, beta2, R_s, a, e = sp.symbols('kappa2 beta2 R_s a e', positive=True)

# Exact RG building blocks
tau2_exact   = (1 - beta2)*(1 - kappa2)
tau_Y2_exact = sp.expand(1 - tau2_exact)

print("tau_Y^2 (exact, generic kappa, beta):")
sp.pprint(tau_Y2_exact)

# Closure at semi-major axis: beta^2 = kappa^2 / 2
tau_Y2_closure = sp.expand(tau_Y2_exact.subs(beta2, kappa2/2))
print("\ntau_Y^2 after closure beta^2 = kappa^2/2:")
sp.pprint(tau_Y2_closure)

# Substitute kappa^2 = R_s / a
tau_Y2_full = sp.expand(tau_Y2_closure.subs(kappa2, R_s/a))
print("\ntau_Y^2 with kappa^2 = R_s/a:")
sp.pprint(tau_Y2_full)

# Full RG precession
dphi_RG = sp.expand(2*sp.pi*tau_Y2_full / (1 - e**2))
print("\nRG precession (full, closed form):")
sp.pprint(dphi_RG)

# GR first-order PN precession
dphi_GR = 3*sp.pi*R_s / (a*(1 - e**2))
print("\nGR first-order PN precession:")
sp.pprint(dphi_GR)

# Algebraic difference RG - GR
diff = sp.simplify(sp.expand(dphi_RG - dphi_GR))
print("\nAlgebraic difference  RG - GR  =")
sp.pprint(diff)

# Same difference expressed as  -2 pi (beta^2 kappa^2) / (1-e^2)
# at the semi-major axis (closure: beta^2 kappa^2 = kappa^4 / 2 = R_s^2/(2 a^2))
predicted = -2*sp.pi*(R_s/(2*a))*(R_s/a) / (1 - e**2)
predicted_simplified = sp.simplify(predicted)
print("\nPredicted second-order term  -2 pi (beta^2 kappa^2)/(1-e^2) =")
sp.pprint(predicted_simplified)

print("\nIdentity check  (RG - GR) - predicted =")
sp.pprint(sp.simplify(diff - predicted))

# -----------------------------------------------------------
# Taylor expansion in R_s/a (alternative verification)
# -----------------------------------------------------------
eps = sp.symbols('eps', positive=True)
dphi_RG_eps = dphi_RG.subs(R_s, eps*R_s)
series_o1 = sp.series(dphi_RG_eps, eps, 0, 2).removeO().subs(eps, 1)
series_o2 = sp.series(dphi_RG_eps, eps, 0, 3).removeO().subs(eps, 1)

print("\nRG precession, first order in (R_s/a):")
sp.pprint(sp.simplify(series_o1))

print("\nRG precession to second order in (R_s/a):")
sp.pprint(sp.simplify(series_o2))

print("\nFirst-order RG  -  GR (first order PN)  =")
sp.pprint(sp.simplify(series_o1 - dphi_GR))

tau_Y^2 (exact, generic kappa, beta):
-β₂⋅κ₂ + β₂ + κ₂

tau_Y^2 after closure beta^2 = kappa^2/2:
    2       
  κ₂    3⋅κ₂
- ─── + ────
   2     2  

tau_Y^2 with kappa^2 = R_s/a:
    2        
  Rₛ     3⋅Rₛ
- ──── + ────
     2   2⋅a 
  2⋅a        

RG precession (full, closed form):
         2                 
     π⋅Rₛ          3⋅π⋅Rₛ  
- ──────────── + ──────────
     2  2    2        2    
  - a ⋅e  + a    - a⋅e  + a

GR first-order PN precession:
  3⋅π⋅Rₛ  
──────────
  ⎛     2⎞
a⋅⎝1 - e ⎠

Algebraic difference  RG - GR  =
       2   
   π⋅Rₛ    
───────────
 2 ⎛ 2    ⎞
a ⋅⎝e  - 1⎠

Predicted second-order term  -2 pi (beta^2 kappa^2)/(1-e^2) =
       2   
   π⋅Rₛ    
───────────
 2 ⎛ 2    ⎞
a ⋅⎝e  - 1⎠

Identity check  (RG - GR) - predicted =
0

RG precession, first order in (R_s/a):
 -3⋅π⋅Rₛ  
──────────
  ⎛ 2    ⎞
a⋅⎝e  - 1⎠

RG precession to second order in (R_s/a):
π⋅Rₛ⋅(Rₛ - 3⋅a)
───────────────
   2 ⎛ 2    ⎞  
  a ⋅⎝e  - 1⎠  

First-order RG  -  GR (first order PN)  =
0


In [2]:
"""
Numerical comparison: WILL RG exact precession vs first-order GR formula
for Mercury's perihelion advance.
"""

import mpmath as mp

mp.mp.dps = 40

# -----------------------------------------------------------
# Physical constants and Mercury orbital parameters
# -----------------------------------------------------------
c          = mp.mpf('299792458')           # m/s (exact)
GM_sun     = mp.mpf('1.32712440018e20')    # m^3/s^2 (IAU 2015)
a_merc     = mp.mpf('5.7909050e10')        # m, semi-major axis
e_merc     = mp.mpf('0.20563593')          # eccentricity (J2000)
T_merc     = mp.mpf('87.9691')             # days, orbital period
days_per_century = mp.mpf('36525')

# Schwarzschild radius of the Sun
R_s = 2*GM_sun / c**2

# Dimensionless parameters
kappa2_a = R_s / a_merc                    # closure at semi-major axis
beta2_a  = kappa2_a / 2

print("Input parameters:")
print(f"  R_s (Sun)    = {mp.nstr(R_s,      12)} m")
print(f"  a (Mercury)  = {mp.nstr(a_merc,   12)} m")
print(f"  e (Mercury)  = {mp.nstr(e_merc,   12)}")
print(f"  kappa^2(a)   = {mp.nstr(kappa2_a, 12)}")
print(f"  beta^2(a)    = {mp.nstr(beta2_a,  12)}")
print(f"  (1 - e^2)    = {mp.nstr(1-e_merc**2, 12)}")

# -----------------------------------------------------------
# WILL RG exact
# -----------------------------------------------------------
tau2_exact   = (1 - beta2_a)*(1 - kappa2_a)
tau_Y2_exact = 1 - tau2_exact
dphi_RG_per_orbit = 2*mp.pi*tau_Y2_exact / (1 - e_merc**2)

# -----------------------------------------------------------
# GR first-order PN
# -----------------------------------------------------------
dphi_GR_per_orbit = 3*mp.pi*R_s / (a_merc*(1 - e_merc**2))

# -----------------------------------------------------------
# Difference and analytical prediction of second-order term
# -----------------------------------------------------------
diff_per_orbit       = dphi_RG_per_orbit - dphi_GR_per_orbit
predicted_per_orbit  = -mp.pi*R_s**2 / (a_merc**2 * (1 - e_merc**2))
residual_per_orbit   = diff_per_orbit - predicted_per_orbit

# Conversion to arcseconds per century
rad_to_arcsec = 180*3600 / mp.pi
orbits_per_century = days_per_century / T_merc

dphi_RG_arcsec_century = dphi_RG_per_orbit * orbits_per_century * rad_to_arcsec
dphi_GR_arcsec_century = dphi_GR_per_orbit * orbits_per_century * rad_to_arcsec
diff_arcsec_century    = diff_per_orbit    * orbits_per_century * rad_to_arcsec
predicted_arcsec_century = predicted_per_orbit * orbits_per_century * rad_to_arcsec

print("\nPer-orbit values (radians):")
print(f"  WILL RG (exact)            = {mp.nstr(dphi_RG_per_orbit,    18)}")
print(f"  GR  (first-order PN)       = {mp.nstr(dphi_GR_per_orbit,    18)}")
print(f"  RG - GR                    = {mp.nstr(diff_per_orbit,        6)}")
print(f"  predicted -pi R_s^2/(a^2(1-e^2)) = {mp.nstr(predicted_per_orbit, 6)}")
print(f"  residual                   = {mp.nstr(residual_per_orbit,     6)}")

print("\nArcseconds per century:")
print(f"  WILL RG (exact)            = {mp.nstr(dphi_RG_arcsec_century, 12)}")
print(f"  GR  (first-order PN)       = {mp.nstr(dphi_GR_arcsec_century, 12)}")
print(f"  RG - GR                    = {mp.nstr(diff_arcsec_century,     6)}")

# Relative size of the omitted term
rel = diff_per_orbit / dphi_RG_per_orbit
print(f"\nRelative size of (RG - GR): {mp.nstr(rel, 6)}")

Input parameters:
  R_s (Sun)    = 2953.2500765 m
  a (Mercury)  = 57909050000.0 m
  e (Mercury)  = 0.20563593
  kappa^2(a)   = 5.09980750246e-8
  beta^2(a)    = 2.54990375123e-8
  (1 - e^2)    = 0.957713864293

Per-orbit values (radians):
  WILL RG (exact)            = 5.01867565337212851e-7
  GR  (first-order PN)       = 5.01867573868639579e-7
  RG - GR                    = -8.53143e-15
  predicted -pi R_s^2/(a^2(1-e^2)) = -8.53143e-15
  residual                   = -1.25735e-41

Arcseconds per century:
  WILL RG (exact)            = 42.9807844914
  GR  (first-order PN)       = 42.980785222
  RG - GR                    = -7.30646e-7

Relative size of (RG - GR): -1.69994e-8
